# 00 — Teste de fumaça do acesso aos dados (Fase 0, passo 0.5)

Caderno temporário, **não faz parte da entrega**. Verifica se as fontes podem ser baixadas por código antes de investir na extração completa. Mostra apenas nomes de colunas e contagens — nenhum registro individual é impresso.

**Localização do repositório.** Garante que o caderno rode a partir de `notebooks/`, inclusive quando aberto isolado (ex.: Google Colab), caso em que o repositório público é clonado. Todos os caminhos seguintes são relativos à raiz.

In [ ]:
# Localiza a raiz do repositório (pasta com requirements.txt e src/appd).
# Se o caderno foi aberto isolado (ex.: Google Colab), clona o repositório
# público na pasta atual. Depois disso, tudo usa caminhos relativos à raiz.
import os
import subprocess
from pathlib import Path

URL_REPOSITORIO = "https://github.com/bsousa7/APPD_26.git"


def localizar_raiz():
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / "requirements.txt").exists() and (pasta / "src" / "appd").exists():
            return pasta
    return None


if localizar_raiz() is None:
    copia = Path("APPD_26")
    if (copia / ".git").exists():
        subprocess.run(["git", "-C", str(copia), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(copia)], check=True)
    os.chdir(copia)
    if localizar_raiz() is None:
        raise RuntimeError(
            f"A cópia em {Path.cwd()} não tem requirements.txt e src/appd. "
            "Apague a pasta APPD_26 e execute esta célula de novo."
        )
os.chdir(localizar_raiz() / "notebooks")
versao = subprocess.run(["git", "log", "-1", "--format=%h %cs"], capture_output=True, text=True).stdout.strip()
print(f"Pasta de trabalho: {Path.cwd().parent.name}/notebooks | versão do repositório: {versao}")

## 1. Ambiente
Instala dependências, define caminhos e aponta o cache do PySUS para `data/raw/pysus` (ignorado pelo Git).

In [ ]:
# Escolhe o arquivo de requisitos conforme o ambiente:
# - Google Colab: requirements-colab.txt (só o que o Colab não traz, sem
#   substituir pandas, ipykernel etc., que o próprio Colab exige);
# - ambiente local: requirements.txt (todas as bibliotecas, versões fixadas).
# O caminho é calculado a partir de notebooks/ e passado já resolvido ao %pip.
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
nome = "requirements-colab.txt" if NO_COLAB else "requirements.txt"
ARQ_REQUISITOS = (Path.cwd().parent / nome).resolve()
if not ARQ_REQUISITOS.exists():
    raise FileNotFoundError(
        f"{nome} não encontrado a partir de {Path.cwd().name}/. "
        "Execute antes a célula 'Localização do repositório' (seção 1)."
    )
print("Instalando", nome)
%pip install -q -r {ARQ_REQUISITOS}

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

RAIZ = Path("..")
sys.path.insert(0, str(RAIZ / "src"))

DIR_RAW = RAIZ / "data" / "raw"
DIR_PROC = RAIZ / "data" / "processed"
DIR_DIC = DIR_PROC / "dicionarios"
DIR_FIG = RAIZ / "outputs" / "figuras"
DIR_TAB = RAIZ / "outputs" / "tabelas"
for d in (DIR_RAW, DIR_PROC, DIR_DIC, DIR_FIG, DIR_TAB):
    d.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option("display.max_columns", 30)
from importlib.metadata import version
print("Python", sys.version.split()[0], "|", ", ".join(
    f"{p} {version(p)}" for p in ["pandas", "numpy", "scipy", "statsmodels",
                                  "scikit-learn", "linearmodels", "differences", "pysus"]))

import pysus
from appd import download as dl
from appd.agregacao import TP_UNID_POLO

dl.configurar_cache(RAIZ)
print("PySUS", pysus.__version__)

O cache do PySUS passa a ficar dentro de `data/raw/`, que é esvaziado na entrega.

## 2. CNES-ST: em que ano o tipo 74 aparece?
**Por quê:** confirmar que o campo `TP_UNID` existe e descobrir a primeira competência com estabelecimentos do tipo 74 (Polo Academia da Saúde). O tipo foi criado pela Portaria SAS/MS nº 24, de 14/01/2014, então é esperado que não haja nenhum em 2012 e 2013.

Lê só `CNES`, `CODUFMUN` e `TP_UNID`: o arquivo completo traz `CPF_CNPJ`, que não deve ser carregado (Política de IA, art. 4º, VI). Usa DF e PE (estado com muitos polos) em dezembro de 2012 a 2016.

In [ ]:
COLS_ST = ["CNES", "CODUFMUN", "TP_UNID"]
contagem = {}
for uf in ["DF", "PE"]:
    for ano in range(2012, 2017):
        st = dl.baixar_datasus("cnes", uf, ano, 12, grupo="ST", colunas=COLS_ST)
        assert set(COLS_ST) <= set(st.columns)
        contagem[(uf, ano)] = int((st["TP_UNID"].str.strip().str.zfill(2) == TP_UNID_POLO).sum())
pd.Series(contagem, name="estabelecimentos tipo 74").unstack(0)

**Resultado:** [PREENCHER após executar: primeiro ano com polos tipo 74 em cada UF]

## 3. SIH-RD, DF, janeiro de 2024
**Por quê:** confirmar os campos `MUNIC_RES`, `DIAG_PRINC` e `ANO_CMPT` e a disponibilidade de 2024.

In [ ]:
rd = dl.baixar_datasus("sih", "DF", 2024, 1, grupo="RD")
print("Colunas:", list(rd.columns)[:40], "...")
print("Campos necessários presentes:", {c: c in rd.columns for c in ["MUNIC_RES", "DIAG_PRINC", "ANO_CMPT", "IDADE", "COD_IDADE"]})
print("Linhas:", len(rd))

**Resultado:** [PREENCHER após executar]

## 4. IBGE: lista de municípios e população (SIDRA)
**Por quê:** confirmar a API de localidades e o número da tabela de estimativas populacionais (provavelmente 6579, variável 9324).

In [ ]:
mun = dl.municipios_ibge()
print("Municípios:", len(mun), "| UFs:", mun["uf"].nunique())
pop = dl.tabela_sidra("6579", "9324", "2015")
print("Colunas SIDRA:", list(pop.columns))
print("Linhas:", len(pop))

**Resultado:** [PREENCHER após executar]

## 5. Registro de acessos e versões

In [ ]:
display(dl.registro_acessos())
%pip show pysus differences linearmodels

Anote as versões e as datas de acesso no `README.md` e no registro de uso de IA.